# BrailleLens — Combined Fingertip YOLO26 (TI1K + Roboflow + Braille)

Train **`yolo26n.pt`** (COCO) on the **merged** fingertip dataset.

### Upload to Google Drive (either folder works):

`MyDrive/BrailleLens_Fingertip_Domain/` **or** `MyDrive/BrailleLens_Fingertip_Combined/`

**Recommended — single zip:** `fingertip_combined_yolo26.zip` (~577 MB)

**Runtime → T4 GPU** → Run all cells.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

# Accept either Drive folder (Domain = where you uploaded fingertip_combined_yolo26.zip)
_DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/BrailleLens_Fingertip_Domain"),
    Path("/content/drive/MyDrive/BrailleLens_Fingertip_Combined"),
]
DRIVE_ROOT = next((p for p in _DRIVE_CANDIDATES if p.is_dir()), _DRIVE_CANDIDATES[0])
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Zip name from build_combined_dataset.py on your PC
_COMBINED_NAMES = [
    "fingertip_combined_yolo26.zip",
    "fingertip_combined_yolo.zip",
]
COMBINED_ZIP = next((DRIVE_ROOT / n for n in _COMBINED_NAMES if (DRIVE_ROOT / n).exists()), DRIVE_ROOT / _COMBINED_NAMES[0])
BASE_ZIP = DRIVE_ROOT / "fingertip_yolo26.zip"
BRAILLE_ZIP = DRIVE_ROOT / "braille_fingertip_yolo.zip"

# Set after unpack in cell 2 — default folder inside the zip
LOCAL_DATA = Path("/content/fingertip_combined_yolo26")
DATA_YAML = LOCAL_DATA / "data.yaml"
RUNS_DIR = DRIVE_ROOT / "runs" / "fingertip_combined"
RUN_NAME = "yolo26n_combined"
BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
EXPORT_PT = DRIVE_ROOT / "yolo26n_fingertip_combined_best.pt"
METRICS_OUT = DRIVE_ROOT / "metrics_summary_combined.json"

print("DRIVE_ROOT   :", DRIVE_ROOT)
print("combined zip :", COMBINED_ZIP.exists(), "->", COMBINED_ZIP)
print("base zip     :", BASE_ZIP.exists())
print("braille zip  :", BRAILLE_ZIP.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT   : /content/drive/MyDrive/BrailleLens_Fingertip_Domain
combined zip : True -> /content/drive/MyDrive/BrailleLens_Fingertip_Domain/fingertip_combined_yolo26.zip
base zip     : False
braille zip  : False


## 1) Install Ultralytics

In [2]:
import os
!pip -q uninstall -y pillow
!pip -q install --no-cache-dir --force-reinstall "pillow>=11.3.0"
!pip -q install -U ultralytics pyyaml opencv-python-headless
try:
    from ultralytics import YOLO
    import torch, ultralytics
    print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 67.8 MB/s eta 0:00:00
ultralytics 8.4.171 | CUDA: True
GPU: Tesla T4


## 2) Prepare dataset on local disk

In [3]:
import shutil
import zipfile
import yaml

def count_imgs(root, split):
    d = root / "images" / split
    return len(list(d.glob("*.*"))) if d.is_dir() else 0

def find_dataset_root():
    """Locate unpacked YOLO folder (data.yaml + images/train)."""
    for cand in (
        Path("/content/fingertip_combined_yolo26"),
        Path("/content/fingertip_combined_yolo"),
        Path("/content/fingertip_yolo26"),
    ):
        if (cand / "data.yaml").exists() and (cand / "images" / "train").is_dir():
            return cand
    hits = list(Path("/content").rglob("data.yaml"))
    for yaml_path in hits:
        root = yaml_path.parent
        if (root / "images" / "train").is_dir():
            return root
    return None

if find_dataset_root() and count_imgs(find_dataset_root(), "train") > 100:
    LOCAL_DATA = find_dataset_root()
    DATA_YAML = LOCAL_DATA / "data.yaml"
    print("Dataset already ready:", LOCAL_DATA)
elif COMBINED_ZIP.exists():
    print("Unpacking", COMBINED_ZIP, "-> /content (may take a few minutes)...")
    with zipfile.ZipFile(COMBINED_ZIP, "r") as zf:
        zf.extractall("/content")
    LOCAL_DATA = find_dataset_root()
    if LOCAL_DATA is None:
        raise FileNotFoundError("Zip unpacked but no data.yaml found under /content")
    DATA_YAML = LOCAL_DATA / "data.yaml"
    print("Found dataset at:", LOCAL_DATA)
elif BASE_ZIP.exists() and BRAILLE_ZIP.exists():
    print("Merging fingertip_yolo26 + braille on Colab...")
    base_root = Path("/content/fingertip_yolo26")
    braille_root = Path("/content/braille_fingertip_yolo")
    with zipfile.ZipFile(BASE_ZIP, "r") as zf:
        zf.extractall("/content")
    with zipfile.ZipFile(BRAILLE_ZIP, "r") as zf:
        zf.extractall("/content")
    LOCAL_DATA = Path("/content/fingertip_combined_yolo26")
    if LOCAL_DATA.exists():
        shutil.rmtree(LOCAL_DATA)
    shutil.copytree(base_root, LOCAL_DATA)
    for split in ("train", "val", "test"):
        for img in (braille_root / "images" / split).glob("*.*"):
            stem = img.stem
            shutil.copy2(img, LOCAL_DATA / "images" / split / f"braille_{img.name}")
            lbl = braille_root / "labels" / split / f"{stem}.txt"
            if lbl.exists():
                shutil.copy2(lbl, LOCAL_DATA / "labels" / split / f"braille_{stem}.txt")
    DATA_YAML = LOCAL_DATA / "data.yaml"
    print("Merged into", LOCAL_DATA)
else:
    raise FileNotFoundError(
        f"Upload fingertip_combined_yolo26.zip to one of:\n"
        f"  {_DRIVE_CANDIDATES[0]}\n"
        f"  {_DRIVE_CANDIDATES[1]}\n"
        f"Current DRIVE_ROOT: {DRIVE_ROOT}\n"
        f"Looking for: {COMBINED_ZIP}"
    )

cfg = {
    "path": str(LOCAL_DATA.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "fingertip"},
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

for split in ("train", "val", "test"):
    print(f"{split:5s}: {count_imgs(LOCAL_DATA, split)} images")

Dataset already ready: /content/drive/MyDrive/BrailleLens/fingertip_yolo26
train: 12537 images
val  : 757 images
test : 306 images


## 3) Train from `yolo26n.pt` (COCO pretrained)

In [4]:
from ultralytics import YOLO
import torch

EPOCHS = 80
IMGSZ = 640
BATCH = 16
PATIENCE = 25
LAST_PT = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

resume = LAST_PT.exists()
print("Resume?", resume)

model = YOLO(str(LAST_PT) if resume else "yolo26n.pt")

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    save=True,
    save_period=1,
    patience=PATIENCE,
    workers=4,
    seed=42,
    plots=True,
    lr0=0.01,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=5.0,
    translate=0.10,
    scale=0.30,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.05,
    close_mosaic=10,
)
print("best.pt:", BEST_PT, BEST_PT.exists())

Resume? True
WARNING ⚠️ 'keras' is deprecated and will be removed in the future.
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/BrailleLens/fingertip_yolo26/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content

## 4) Evaluate + export

In [5]:
import json

import shutil
from ultralytics import YOLO

if not BEST_PT.exists():
    raise FileNotFoundError("Training not finished — no best.pt")

model = YOLO(str(BEST_PT))

def eval_split(split):
    m = model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=0 if torch.cuda.is_available() else "cpu", split=split)
    p, r = float(m.box.mp), float(m.box.mr)
    return {
        "precision": round(p, 4),
        "recall": round(r, 4),
        "f1": round(2*p*r/(p+r+1e-9), 4),
        "map50": round(float(m.box.map50), 4),
        "map50_95": round(float(m.box.map), 4),
    }

summary = {
    "model": "yolo26n",
    "training": "TI1K + Roboflow + Braille_fingertip combined",
    "base_weights": "yolo26n.pt",
    "val": eval_split("val"),
    "test": eval_split("test"),
}
METRICS_OUT.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

shutil.copy2(BEST_PT, EXPORT_PT)
print("\nDownload:", EXPORT_PT)
print("Metrics:", METRICS_OUT)

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.0 ms, read: 26.0±13.6 MB/s, size: 25.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/BrailleLens/fingertip_yolo26/labels/val.cache... 757 images, 203 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 757/757 264.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 4.7it/s 10.1s
                   all        757        894      0.977      0.893      0.947        0.7
Speed: 1.7ms preprocess, 4.8ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO